# Day 3: Data Visualization & EDA

Comprehensive exploratory data analysis with professional visualizations.

## 📋 Objectives
- Create distribution plots for all features
- Visualize relationships between features and target
- Generate correlation heatmaps
- Create publication-ready plots

In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
df = pd.read_csv('../../day12/data/student_scores_cleaned_encoded.csv')

# Professional styling
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_context('notebook', font_scale=1.1)
sns.set_palette('viridis')

print(f"Dataset shape: {df.shape}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# 1. Target Variable Distribution
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Assuming last column or 'score' related is target
target_candidates = [c for c in df.columns if 'score' in c.lower() or 'mark' in c.lower() or 'grade' in c.lower()]
target_col = target_candidates[0] if target_candidates else df.select_dtypes(include=[np.number]).columns[-1]

# Histogram with KDE
sns.histplot(data=df, x=target_col, kde=True, bins=30, ax=ax1, color='#2E86AB')
ax1.set_title(f'Distribution of {target_col}', fontsize=14, fontweight='bold')
ax1.set_xlabel(target_col)
ax1.set_ylabel('Frequency')

# Box plot
sns.boxplot(data=df, y=target_col, ax=ax2, color='#A23B72')
ax2.set_title(f'Box Plot - {target_col}', fontsize=14, fontweight='bold')
ax2.set_ylabel(target_col)

plt.tight_layout()
plt.savefig('../../day12/plots/target_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Target column: {target_col}")
print(f"Mean: {df[target_col].mean():.2f}")
print(f"Median: {df[target_col].median():.2f}")
print(f"Std: {df[target_col].std():.2f}")

In [ ]:
# 2. Feature Distributions (Grid)
numeric_cols = df.select_dtypes(include=[np.number]).columns
feature_cols = [c for c in numeric_cols if c != target_col]

n_cols = 3
n_rows = int(np.ceil(len(feature_cols) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes

for idx, col in enumerate(feature_cols):
    sns.histplot(data=df, x=col, kde=True, ax=axes[idx], bins=20, color='#F18F01')
    axes[idx].set_title(f'{col}', fontsize=12)
    axes[idx].set_xlabel('')

# Hide empty subplots
for idx in range(len(feature_cols), len(axes)):
    axes[idx].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../day12/plots/feature_distributions.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 3. Correlation Heatmap (Enhanced)
corr_matrix = df[numeric_cols].corr()

plt.figure(figsize=(12, 10))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, 
            mask=mask,
            annot=True,
            fmt='.2f',
            cmap='RdBu_r',
            center=0,
            square=True,
            linewidths=0.5,
            cbar_kws={'shrink': 0.8, 'label': 'Correlation Coefficient'})
plt.title('Feature Correlation Matrix (Lower Triangle)', fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig('../../day12/plots/correlation_heatmap.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 4. Target vs Features - Scatter Plots
n_features = len(feature_cols)
n_cols = 3
n_rows = int(np.ceil(n_features / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, 4*n_rows))
axes = axes.flatten() if n_rows > 1 else [axes] if n_cols == 1 else axes

for idx, col in enumerate(feature_cols):
    sns.scatterplot(data=df, x=col, y=target_col, ax=axes[idx], alpha=0.6, s=50, color='#C73E1D')
    axes[idx].set_title(f'{target_col} vs {col}', fontsize=12)
    axes[idx].set_xlabel(col)
    axes[idx].set_ylabel(target_col)
    
    # Add correlation annotation
    corr = df[col].corr(df[target_col])
    axes[idx].annotate(f'r = {corr:.3f}', xy=(0.05, 0.95), xycoords='axes fraction',
                       fontsize=11, fontweight='bold',
                       bbox=dict(boxstyle='round,pad=0.3', facecolor='white', alpha=0.8))

for idx in range(n_features, len(axes)):
    axes[idx].set_visible(False)

plt.suptitle(f'{target_col} vs All Features', fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../../day12/plots/target_vs_features.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
# 5. Pair Plot (Sample for performance)
# Select top correlated features with target
correlations = df[feature_cols].corrwith(df[target_col]).abs().sort_values(ascending=False)
top_features = correlations.head(5).index.tolist()
plot_cols = top_features + [target_col]

sns.pairplot(df[plot_cols], diag_kind='kde', plot_kws={'alpha': 0.6, 's': 30})
plt.suptitle('Pair Plot - Top Correlated Features', fontsize=16, fontweight='bold', y=1.02)
plt.savefig('../../day12/plots/pairplot.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Top correlated features with {target_col}:")
for feat, corr in correlations.head(5).items():
    print(f"  {feat}: {corr:.4f}")

In [ ]:
# 6. Categorical Feature Analysis (if any)
cat_cols = df.select_dtypes(include=['object', 'category', 'bool']).columns
encoded_cat_cols = [c for c in df.columns if any(cat in c for cat in ['_', 'gender', 'dept', 'department']) and df[c].nunique() < 10]

if len(encoded_cat_cols) > 0:
    n_cats = len(encoded_cat_cols)
    n_cols = min(3, n_cats)
    n_rows = int(np.ceil(n_cats / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(6*n_cols, 4*n_rows))
    axes = axes.flatten() if n_rows * n_cols > 1 else [axes]
    
    for idx, col in enumerate(encoded_cat_cols):
        if df[col].dtype in ['uint8', 'int64', 'float64'] and df[col].nunique() == 2:
            # Binary encoded column - box plot
            sns.boxplot(data=df, x=col, y=target_col, ax=axes[idx])
            axes[idx].set_title(f'{target_col} by {col}')
        else:
            # Other categorical
            sns.violinplot(data=df, x=col, y=target_col, ax=axes[idx])
            axes[idx].set_title(f'{target_col} by {col}')
    
    for idx in range(n_cats, len(axes)):
        axes[idx].set_visible(False)
    
    plt.tight_layout()
    plt.savefig('../../day12/plots/categorical_analysis.png', dpi=300, bbox_inches='tight')
    plt.show()
else:
    print("No categorical columns found for analysis")

## 📝 Summary

- Target variable distribution analyzed
- All feature distributions visualized
- Correlation matrix with masking for clarity
- Target vs feature scatter plots with correlations
- Pair plot for top correlated features
- Categorical feature analysis (if applicable)

---
*Next: [04_model_training.ipynb](04_model_training.ipynb)*